In [6]:
# import packages
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import shap
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

In [7]:
seed = 2724

### Import data

In [8]:
DF_PATH = "mod04_data/sample.csv"
df = pd.read_csv(DF_PATH)

NameError: name 'pd' is not defined

### Separate data by independent (X) and dependent (y) variables

In [ ]:
X = df[["income", "education_years", "zipcode_score"]]
y = df["target"]

NameError: name 'df' is not defined

### Split the data into a _training_ set (to build a model) and _test_ set (to validate a model)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed
)

### Build a model on the training set

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=seed
)
model.fit(X_train, y_train)

### Use SHAP to explain the model on test data

In [ ]:
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)

This will allow us to see which variables are most important to predicting the outcome.

In [ ]:
shap.plots.bar(shap_values)

### Import the `group` variable, which was **not** used in training this model.

In [ ]:
X_test_with_group = X_test.copy()
X_test_with_group["group"] = df.loc[X_test.index, "group"]

### Look at the difference in SHAP values between the two groups across the variables used in the model.

In [ ]:
shap_df = pd.DataFrame(shap_values.values, columns=X_test.columns)
shap_df["group"] = X_test_with_group["group"].values

shap_df.groupby("group").mean()

### Let's put `group` and `zipcode_score` in the same plot

In [ ]:
def plot_shap(var):
    # Extract SHAP values for the feature
    shap_var = shap_values[:, var].values

    # Plot the values of each group using different colors
    plt.figure()
    plt.scatter(
        X_test[var],
        shap_var,
        c=X_test_with_group["group"]
    )
    plt.xlabel(var)
    plt.ylabel(f"SHAP value for {var}")
    plt.title("Proxy feature impact by group")
    plt.show()

plot_shap("zipcode_score")

# Discussion Questions

### What is a _SHAP_ (or Shapley) value? 

A SHAP (Shapley) value measures how much one feature contributes to a particular model prediction relative to the model's baseline prediction. It is based on Shapley values from cooperative game theory and averages the feature's marginal contribution across different possible combinations of features. A positive SHAP value pushes the prediction higher, while a negative SHAP value pushes it lower

### Suppose you built this model and then it is peer reviewed by another entity. If the reviewer asks whether you used the variable `group` in your model, what would your answer be?

Suppose you built this model and then it is peer reviewed by another entity. If the reviewer asks whether you used the variable group in your model, what would your answer be?

### If the reviewer asks whether the outcome of your model is correlated with `group`, what would your answer be?

No. The model was trained only with income, education_years, and zipcode_score; group was not included as an input feature. The group variable was added back only after training so that the model's behavior could be audited across demographic groups. However, excluding group does not by itself guarantee that the model is unbiased because another feature can act as a proxy for it

### Construct a "proxy feature impact by group" plot for `income`. How is this plot different from the one for `zipcode_score`?

In [ ]:
plot_shap("income")

NameError: name 'plot_shap' is not defined

The income plot shows the two groups largely overlapping across both income values and their SHAP effects. The groups have nearly identical mean incomes (about $59,948 for group 0 and $59,971 for group 1) and nearly identical mean income SHAP values (about +1.09 and +1.02). In contrast, the zipcode_score plot shows clear separation between the groups: group 0 generally has higher zipcode scores and positive SHAP effects, while group 1 generally has lower zipcode scores and negative SHAP effects. This indicates that income is not acting as a meaningful proxy for group, whereas zipcode_score is.

### If, instead, you were the **reviewer**, what other questions might you ask the person who built this model? Give at least two.

As a reviewer, I would ask:  
1. How were the input features selected, and were they tested to determine whether any of them act as proxies for protected demographic characteristics?
2. How do prediction accuracy, error size, and the distribution of predictions compare across the demographic groups and relevant intersectional subgroups?
3. Is the training data representative of the population in which the model will be used, and could the target variable or its labels contain historical bias?
4. What bias-mitigation, human-review, appeal, and ongoing-monitoring procedures will be used before and after the model is deployed?Type your answer here.